# 🚀 QwerySmith 1.1: Live Interactive Database Agent Demo

Welcome to **QwerySmith 1.1**! In this notebook, we connect the fine-tuned **QwerySmith 1.1** model directly to a real-world enterprise SQLite database pulled from the web (**The Chinook Database** — 11 relational tables covering customers, invoices, tracks, albums, artists, genres, and playlists).

[![Hugging Face](https://img.shields.io/badge/%F0%9F%A4%97%20Model-Cyrax321%2FQwerySmith--1.1-blue)](https://huggingface.co/Cyrax321/QwerySmith-1.1)
[![GitHub](https://img.shields.io/badge/GitHub-QwerySmith--1.0-black)](https://github.com/Cyrax321/QwerySmith-1.0)
[![Paper](https://img.shields.io/badge/Paper-PDF-red)](https://drive.google.com/file/d/1sN1eVn7LpOi6cLEI1euxOT2cByBoXLlg/view?usp=sharing)

### ⚡ Hardware Requirements
A free Google Colab **T4 GPU** is plenty. Make sure your runtime is set to **GPU** (`Runtime > Change runtime type > T4 GPU`).

## Step 1: Install Dependencies
Install Unsloth and TRL for fast 4-bit inference (~40 seconds).

In [ ]:
!pip install -q "unsloth[colab-new] @ git+https://github.com/unslothai/unsloth.git" trl peft

## Step 2: Download the Agent & a Real Database from the Web
We pull:
1. `agent.py` directly from GitHub.
2. `chinook.db` — a live SQLite digital media store with 11 relational tables.

In [ ]:
# Download latest QwerySmith Agent
!curl -sL -o agent.py https://raw.githubusercontent.com/Cyrax321/QwerySmith-1.0/main/agent.py

# Download the real Chinook database (Albums, Artists, Customers, Invoices, Tracks, etc.)
!curl -sL -o chinook.db https://raw.githubusercontent.com/lerocha/chinook-database/master/ChinookDatabase/DataSources/Chinook_Sqlite.sqlite

import sqlite3
conn = sqlite3.connect("chinook.db")
cur = conn.cursor()
cur.execute("SELECT name FROM sqlite_master WHERE type='table' AND name NOT LIKE 'sqlite_%';")
tables = [r[0] for r in cur.fetchall()]
print(f"✅ Successfully loaded chinook.db with {len(tables)} tables:")
for t in tables:
    cur.execute(f"SELECT count(*) FROM {t}; ")
    print(f"  • {t:<15}: {cur.fetchone()[0]:>5} rows")
conn.close()

## Step 3: Initialize QwerySmith 1.1
Loads the adapter weights from HuggingFace (`Cyrax321/QwerySmith-1.1`) directly into GPU VRAM in fast 4-bit precision.

In [ ]:
from agent import QwerySmithAgent

agent = QwerySmithAgent(model_path="Cyrax321/QwerySmith-1.1")
print("⚡ QwerySmith 1.1 is live and ready!")

## Step 4: Ask Queries Against the Real Database!
Try running some complex relational questions across tables:

In [ ]:
# Example 1: Multi-table aggregation (Artist + Album)
res1 = agent.query("chinook.db", "Which 5 artists have the most albums in the store?")
print("🤖 Human Explanation:", res1.get("human_answer"))
print("\n🔍 Executed SQL:", res1.get("sql"))

In [ ]:
# Example 2: Financial calculation (Customer + Invoice)
res2 = agent.query("chinook.db", "Find the top 5 customers who spent the most total money on invoices.")
print("🤖 Human Explanation:", res2.get("human_answer"))
print("\n🔍 Executed SQL:", res2.get("sql"))

In [ ]:
# Example 3: Filter with string matching and grouping (Tracks + Genre)
res3 = agent.query("chinook.db", "What are the top 3 most common music genres by number of tracks?")
print("🤖 Human Explanation:", res3.get("human_answer"))
print("\n🔍 Executed SQL:", res3.get("sql"))

## Step 5: Launch the Interactive Live Chat Loop
Run this cell to start an ongoing terminal chat session! You can type anything:
- 'hey, how are you?'
- 'What is an inner join vs left join in SQL?'
- 'Show me the 3 longest tracks in milliseconds and their unit prices'
- Type `:tables` or `:schema` for database introspection
- Type `exit` to quit.

In [ ]:
agent.interactive_chat(db_path="chinook.db")